# English all-token pooling — RF
Three experiments in one notebook; one cumulative CSV for RF.

**Status:** New experiment; no full-grid results have been generated.
Run the definition/data cells once, then the three experiment cells in order (or independently after initialization).
Completed representations resume from this algorithm's CSV without duplicate baseline rows.

## Methods
- Original English dataset/model/cleaning: `34_self_contained_bert_mean_mean_cls3_lr_svm.ipynb`.
- Arabic reference: `experiment_rf_l2_all_tokens_greater_than_k_baselines.py`, the lower/upper-band runner, and `src/classifier_search.py` / `src/evaluation.py`.
- Fixed Arabic settings: seed 42, batch 32, maximum length **320**, 10 shuffled stratified training folds, fold-local StandardScaler for LR/SVM, no scaler for RF; no new parameter grid.
- English BERT replaces MARBERT, and the original English cleaning/stratified 80/20 split replaces the Arabic official split.
- No project imports, edits, installs, downloads, saved embeddings, or separate checkpoint files.
- Final-layer CLS and MEAN baselines; filtered methods are **mean only**, without CLS concatenation.

### Filtering convention
Token L2 norms are standardized **within each sentence**, using population standard deviation over **all attended tokens, including CLS/SEP and other special tokens**. Padding is excluded.
`mean_lt_k` keeps `abs(z) < k`; `mean_gt_k` keeps `abs(z) > k`; combined keeps `lower < abs(z) < upper`.
All 4 × 11 lower/upper combinations are tested. Unlike the old inclusive Arabic K-grid (`<=`), the requested LT operator is **strict**.
If nothing passes, fall back to the full attended-token mean, matching the Arabic band/GT runner, and report the fallback count.
Special tokens are eligible, not forced to survive the filter.

### Evaluation
Each training fold's threshold is optimized on that fold's validation labels, exactly as in the Arabic experiments; these threshold-tuned CV scores are **not nested/unbiased estimates**.
The final threshold comes from pooled training OOF scores; test labels are used only for reporting.
Do not select K using held-out test scores.

### Output
`results/english_all_tokens_rf_results.csv` contains all 61 unique methods, test metrics, CV mean/std, the 10 fold records as JSON, parameters, data/setup provenance, token counts, and fallback counts.
The file is updated after **each representation**, and is complete after all three experiment cells. Run only one instance of this algorithm notebook at a time.


## 1. Imports and fixed experiment settings
Use your existing GPU environment; this cell installs nothing. The cached `bert-base-uncased` model is used locally.


In [1]:
from __future__ import annotations

import gc
import hashlib
import html
import io
import json
import os
import random
import re
import tempfile
import time
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import sklearn
import transformers
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from transformers import AutoModel, AutoTokenizer

# Fixed settings from the successful Arabic all-token experiments.
SEED = 42
MAX_LENGTH = 320
BATCH_SIZE = 32
CV_FOLDS = 10
TEST_SIZE = 0.20
MODEL_NAME = "bert-base-uncased"
LT_K_VALUES = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5]
GT_K_VALUES = [0.0001, 0.001, 0.01, 0.1]
CSV_NAME = "English Sarcasm.csv"
ORIGINAL_ZIP = Path(os.environ.get("MFMP_ENGLISH_ZIP", "data/raw/english/archive.zip"))
EXTRACTED_CSV = Path(os.environ.get("MFMP_ENGLISH_CSV", "data/raw/english/English Sarcasm.csv"))

# Works when launched either from the project root or from notebooks/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "experiments").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from this repository's root or experiments folder.")

CLASSIFIER = "RF"
PARAMETERS = {"n_estimators": 400, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "max_features": "sqrt", "class_weight": "balanced_subsample"}
RESULTS_PATH = PROJECT_ROOT / "results" / f"english_all_tokens_{CLASSIFIER.lower()}_results.csv"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Algorithm:", CLASSIFIER, "| parameters:", PARAMETERS)
print("Model:", MODEL_NAME, "| max length:", MAX_LENGTH, "| batch:", BATCH_SIZE)
print("Device:", DEVICE, "| CV folds:", CV_FOLDS, "| seed:", SEED)
print("One cumulative results file:", RESULTS_PATH)


<PYTHON_ENV>\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Algorithm: RF | parameters: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
Model: bert-base-uncased | max length: 320 | batch: 32
Device: cuda | CV folds: 10 | seed: 42
One cumulative results file: <PROJECT_ROOT>\results\english_all_tokens_rf_results.csv


## 2. Load and prepare the English dataset
Reads the original ZIP if present, otherwise the verified extracted English CSV. Does not change either source.


In [2]:
# Read only the known English dataset; never use the Arabic CSVs.
if ORIGINAL_ZIP.is_file():
    with zipfile.ZipFile(ORIGINAL_ZIP) as archive:
        csv_bytes = archive.read(CSV_NAME)
    DATA_SOURCE = f"{ORIGINAL_ZIP}::{CSV_NAME}"
elif EXTRACTED_CSV.is_file():
    csv_bytes = EXTRACTED_CSV.read_bytes()
    DATA_SOURCE = str(EXTRACTED_CSV)
else:
    raise FileNotFoundError(
        f"English dataset not found. Restore {ORIGINAL_ZIP} (containing {CSV_NAME}) "
        f"or {EXTRACTED_CSV}. No data was downloaded or substituted."
    )
DATA_SHA256 = hashlib.sha256(csv_bytes).hexdigest()
raw_df = pd.read_csv(io.BytesIO(csv_bytes))
if not {"Tweet", "Label"}.issubset(raw_df.columns):
    raise ValueError("The English CSV must have Tweet and Label columns.")

# The English cleaning and deduplication below are copied from notebook 34.
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")

def clean_tweet(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text

df = raw_df[["Tweet", "Label"]].copy()
df["clean_text"] = df["Tweet"].map(clean_tweet)
df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
df = df.dropna(subset=["clean_text", "Label"])
df = df[df["clean_text"].str.len() > 0]
df = df[df["Label"].isin([0, 1])]
df["Label"] = df["Label"].astype(int)
df = df.drop_duplicates(subset=["clean_text", "Label"]).reset_index(drop=True)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["Label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df["Label"].to_numpy()
y_test = test_df["Label"].to_numpy()
if set(np.unique(y_train)) != {0, 1}:
    raise ValueError("Both binary classes 0 and 1 are required.")
if np.bincount(y_train).min() < CV_FOLDS:
    raise ValueError("There are too few training examples per class for 10-fold CV.")

# Every representation and algorithm uses the same training folds.
FOLD_INDICES = list(StratifiedKFold(
    n_splits=CV_FOLDS, shuffle=True, random_state=SEED,
).split(np.zeros(len(y_train)), y_train))
print("Data source:", DATA_SOURCE)
print("Raw / cleaned rows:", len(raw_df), "/", len(df))
print("Train / test rows:", len(train_df), "/", len(test_df))
display(pd.DataFrame({
    "train": train_df["Label"].value_counts().sort_index(),
    "test": test_df["Label"].value_counts().sort_index(),
}))


Data source: data/raw/english/English Sarcasm.csv
Raw / cleaned rows: 4553 / 4546
Train / test rows: 3636 / 910


,train,test
Label,,
0,2836,710
1,800,200


## 3. Representation rules and all-token extraction
61 rules, sentence-local L2 z-scores, strict bounds, and counted empty-filter fallbacks.


In [3]:
def rule(name, kind, lower_k=None, upper_k=None):
    return {"method": name, "kind": kind, "lower_k": lower_k, "upper_k": upper_k}


BASELINE_RULES = [rule("cls", "cls"), rule("mean", "mean")]
LT_RULES = [rule(f"mean_lt_{k:g}", "mean_lt_k", upper_k=k) for k in LT_K_VALUES]
GT_RULES = [rule(f"mean_gt_{k:g}", "mean_gt_k", lower_k=k) for k in GT_K_VALUES]
BAND_RULES = [
    rule(f"mean_gt_{lower:g}_lt_{upper:g}", "mean_gt_k_lt_k", lower, upper)
    for lower in GT_K_VALUES for upper in LT_K_VALUES
]
ALL_RULES = BASELINE_RULES + LT_RULES + GT_RULES + BAND_RULES
RULE_INDEX = {r["method"]: r for r in ALL_RULES}
assert len(RULE_INDEX) == len(ALL_RULES) == 61
assert len(LT_RULES) == 11 and len(GT_RULES) == 4 and len(BAND_RULES) == 44


# Same sentence-local population standard deviation and mean pooling as Arabic.
def l2_z_scores(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    valid_float = attention_mask.bool().float()
    counts = torch.clamp(valid_float.sum(dim=1, keepdim=True), min=1.0)
    token_l2 = torch.linalg.vector_norm(hidden_states, ord=2, dim=-1)
    mean_l2 = torch.sum(token_l2 * valid_float, dim=1, keepdim=True) / counts
    centered_l2 = (token_l2 - mean_l2) * valid_float
    variance_l2 = torch.sum(centered_l2 * centered_l2, dim=1, keepdim=True) / counts
    std_l2 = torch.sqrt(torch.clamp(variance_l2, min=0.0))
    return (token_l2 - mean_l2) / torch.clamp(std_l2, min=1e-12)


def mean_pool(hidden_states: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    expanded_mask = mask.unsqueeze(-1).expand(hidden_states.size()).float()
    counts = torch.clamp(expanded_mask.sum(dim=1), min=1e-9)
    return torch.sum(hidden_states * expanded_mask, dim=1) / counts


def mask_for_rule(abs_z, valid_mask, item):
    if item["kind"] in {"cls", "mean"}:
        return valid_mask, 0
    keep = valid_mask.clone()
    if item["lower_k"] is not None:
        keep &= abs_z > item["lower_k"]
    if item["upper_k"] is not None:
        keep &= abs_z < item["upper_k"]
    has_tokens = keep.any(dim=1, keepdim=True)
    fallback_count = int((~has_tokens).sum().item())
    # Same empty-filter fallback as the Arabic all-token band/greater-than runs.
    return torch.where(has_tokens, keep, valid_mask), fallback_count


tokenizer = None
bert_model = None


def load_english_model():
    global tokenizer, bert_model
    if bert_model is None:
        # Reuse the existing BERT cache. No model downloads or setup changes.
        new_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
        new_model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True).to(DEVICE)
        new_model.eval()
        tokenizer, bert_model = new_tokenizer, new_model
    return tokenizer, bert_model


def extract_representations(texts, items, split_name):
    tok, model = load_english_model()
    texts = list(texts)
    # Final vectors only, not a cache of every token's hidden states.
    vectors = {
        item["method"]: np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
        for item in items
    }
    stats = {
        item["method"]: dict(total_valid_tokens=0, total_special_tokens=0,
                            included_tokens=0, included_special_tokens=0,
                            excluded_tokens=0, excluded_special_tokens=0,
                            filter_fallback_texts=0)
        for item in items
    }
    special_ids = set(tok.all_special_ids)
    print(f"Extracting {split_name}: {len(texts)} texts, {len(items)} representations")
    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            encoded = tok(
                texts[start:start + BATCH_SIZE], padding=True, truncation=True,
                max_length=MAX_LENGTH, return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            hidden = model(**encoded).last_hidden_state
            valid = encoded["attention_mask"].bool()
            # Special tokens participate in the statistics and filtering.
            # Only positions with attention_mask=0 (padding) are excluded.
            abs_z = l2_z_scores(hidden, encoded["attention_mask"]).abs()
            special = torch.zeros_like(valid)
            for special_id in special_ids:
                special |= encoded["input_ids"] == special_id
            special &= valid
            total_valid = int(valid.sum().item())
            total_special = int(special.sum().item())
            stop = start + hidden.shape[0]
            for item in items:
                keep, fallback = mask_for_rule(abs_z, valid, item)
                if item["kind"] == "cls":
                    pooled = hidden[:, 0, :]
                    # Counts reflect the single vector actually used by CLS.
                    keep = torch.zeros_like(valid)
                    keep[:, 0] = valid[:, 0]
                else:
                    pooled = mean_pool(hidden, keep)
                vectors[item["method"]][start:stop] = pooled.cpu().numpy().astype(np.float32)
                included = int(keep.sum().item())
                included_special = int((keep & special).sum().item())
                counts = stats[item["method"]]
                counts["total_valid_tokens"] += total_valid
                counts["total_special_tokens"] += total_special
                counts["included_tokens"] += included
                counts["included_special_tokens"] += included_special
                counts["excluded_tokens"] += total_valid - included
                counts["excluded_special_tokens"] += total_special - included_special
                counts["filter_fallback_texts"] += fallback
            if stop == len(texts) or (start // BATCH_SIZE + 1) % 20 == 0:
                print(f"  {split_name}: {stop}/{len(texts)}")
    for values in vectors.values():
        if not np.isfinite(values).all():
            raise ValueError("Non-finite embedding values; no results were saved.")
    return vectors, stats


print("Representations: 2 baselines + 11 LT + 4 GT + 44 bands = 61")


Representations: 2 baselines + 11 LT + 4 GT + 44 bands = 61


## 4. Fixed classifier and Arabic CV/threshold procedure
Metric and threshold functions are copied from the Arabic evaluation code. No hyperparameter search is added.


In [4]:
def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: list[int] | None = None,
) -> dict[str, Any]:
    labels = labels or [0, 1]
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    per_class_precision, per_class_recall, per_class_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "precision_class_0": per_class_precision[0],
        "recall_class_0": per_class_recall[0],
        "f1_class_0": per_class_f1[0],
        "precision_class_1": per_class_precision[1],
        "recall_class_1": per_class_recall[1],
        "f1_class_1": per_class_f1[1],
    }


def select_best_threshold(
    y_true: np.ndarray,
    scores: np.ndarray,
    verbose: bool = True,
) -> tuple[float, dict[str, Any]]:
    """Select threshold by macro F1, accuracy, macro recall, then class-1 recall."""
    y_true = np.asarray(y_true).ravel()
    scores = np.asarray(scores).ravel()
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must contain the same number of samples.")
    if len(y_true) == 0:
        raise ValueError("Threshold selection requires at least one sample.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("Threshold selection requires binary labels encoded as 0 and 1.")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = y_true[order]
    unique_scores, first_indices = np.unique(sorted_scores, return_index=True)

    total_positive = int(np.sum(y_true == 1))
    total_negative = int(np.sum(y_true == 0))
    tp = total_positive
    fp = total_negative
    fn = 0
    tn = 0

    def metric_row(threshold: float) -> dict[str, Any]:
        precision_0 = tn / (tn + fn) if tn + fn else 0.0
        recall_0 = tn / (tn + fp) if tn + fp else 0.0
        f1_0 = (
            2 * precision_0 * recall_0 / (precision_0 + recall_0)
            if precision_0 + recall_0
            else 0.0
        )
        precision_1 = tp / (tp + fp) if tp + fp else 0.0
        recall_1 = tp / (tp + fn) if tp + fn else 0.0
        f1_1 = (
            2 * precision_1 * recall_1 / (precision_1 + recall_1)
            if precision_1 + recall_1
            else 0.0
        )
        present_class_0 = total_negative > 0 or tn + fn > 0
        present_class_1 = total_positive > 0 or tp + fp > 0
        present_count = int(present_class_0) + int(present_class_1)
        macro_precision = (
            precision_0 * present_class_0 + precision_1 * present_class_1
        ) / present_count
        macro_recall = (
            recall_0 * present_class_0 + recall_1 * present_class_1
        ) / present_count
        macro_f1 = (f1_0 * present_class_0 + f1_1 * present_class_1) / present_count
        return {
            "threshold": float(threshold),
            "accuracy": (tn + tp) / len(y_true),
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "precision_class_0": precision_0,
            "recall_class_0": recall_0,
            "f1_class_0": f1_0,
            "precision_class_1": precision_1,
            "recall_class_1": recall_1,
            "f1_class_1": f1_1,
        }

    best = metric_row(unique_scores[0])
    best_key = (
        best["macro_f1"],
        best["accuracy"],
        best["macro_recall"],
        best["recall_class_1"],
    )

    for unique_index in range(len(unique_scores) - 1):
        start = first_indices[unique_index]
        end = first_indices[unique_index + 1]
        removed_labels = sorted_labels[start:end]
        removed_positive = int(np.sum(removed_labels == 1))
        removed_negative = int(np.sum(removed_labels == 0))
        tp -= removed_positive
        fn += removed_positive
        fp -= removed_negative
        tn += removed_negative

        threshold = (unique_scores[unique_index] + unique_scores[unique_index + 1]) / 2.0
        row = metric_row(threshold)
        row_key = (
            row["macro_f1"],
            row["accuracy"],
            row["macro_recall"],
            row["recall_class_1"],
        )
        if row_key > best_key:
            best = row
            best_key = row_key

    if verbose:
        print("Best validation threshold:")
        print(json.dumps(best, indent=2))
    return float(best["threshold"]), best


def build_estimator():
    # Exact estimator/scaler choices of the corresponding Arabic experiments.
    if CLASSIFIER == "LR":
        classifier = LogisticRegression(random_state=SEED, **PARAMETERS)
        return Pipeline([("scaler", StandardScaler()), ("classifier", classifier)])
    if CLASSIFIER == "SVM":
        classifier = LinearSVC(dual=False, random_state=SEED, **PARAMETERS)
        return Pipeline([("scaler", StandardScaler()), ("classifier", classifier)])
    if CLASSIFIER == "RF":
        return RandomForestClassifier(random_state=SEED, n_jobs=-1, **PARAMETERS)
    raise ValueError(CLASSIFIER)


def positive_scores(estimator, values):
    if hasattr(estimator, "predict_proba"):
        positive_index = int(np.flatnonzero(estimator.classes_ == 1)[0])
        return estimator.predict_proba(values)[:, positive_index], "probability_class_1"
    return np.asarray(estimator.decision_function(values)).ravel(), "decision_score"


def score_predictions(labels, predictions):
    return {
        **calculate_metrics(labels, predictions),
        "tn": int(np.sum((labels == 0) & (predictions == 0))),
        "fp": int(np.sum((labels == 0) & (predictions == 1))),
        "fn": int(np.sum((labels == 1) & (predictions == 0))),
        "tp": int(np.sum((labels == 1) & (predictions == 1))),
        "correct": int(np.sum(labels == predictions)),
        "incorrect": int(np.sum(labels != predictions)),
    }


def evaluate_representation(item, x_train, x_test):
    started = time.perf_counter()
    base = build_estimator()
    oof_scores = np.empty(len(y_train), dtype=float)
    fold_rows = []
    for fold_number, (fit_index, val_index) in enumerate(FOLD_INDICES, 1):
        estimator = clone(base)
        # LR/SVM scaler fits only within this training fold; RF has no scaler.
        estimator.fit(x_train[fit_index], y_train[fit_index])
        scores, score_type = positive_scores(estimator, x_train[val_index])
        oof_scores[val_index] = scores
        threshold, _ = select_best_threshold(y_train[val_index], scores, verbose=False)
        metrics = score_predictions(y_train[val_index], (scores >= threshold).astype(int))
        fold_rows.append({
            "fold": fold_number, "train_size": len(fit_index),
            "validation_size": len(val_index), "best_threshold": threshold, **metrics,
        })
        print(f"  {item['method']} fold {fold_number}/{CV_FOLDS}: "
              f"macro_f1={metrics['macro_f1']:.6f}")

    threshold, _ = select_best_threshold(y_train, oof_scores, verbose=False)
    oof_metrics = score_predictions(y_train, (oof_scores >= threshold).astype(int))
    final_estimator = clone(base)
    final_estimator.fit(x_train, y_train)
    test_scores, score_type = positive_scores(final_estimator, x_test)
    test_metrics = score_predictions(y_test, (test_scores >= threshold).astype(int))
    folds = pd.DataFrame(fold_rows)
    return {
        **item, "classifier": CLASSIFIER, "parameters": json.dumps(PARAMETERS, sort_keys=True),
        **{f"param_{key}": value for key, value in PARAMETERS.items()},
        "cv_macro_f1_mean": float(folds["macro_f1"].mean()),
        "cv_macro_f1_std": float(folds["macro_f1"].std(ddof=1)),
        "cv_accuracy_mean": float(folds["accuracy"].mean()),
        "cv_accuracy_std": float(folds["accuracy"].std(ddof=1)),
        "fold_threshold_mean": float(folds["best_threshold"].mean()),
        "fold_threshold_std": float(folds["best_threshold"].std(ddof=1)),
        "best_threshold": threshold,
        "threshold_source": "pooled_out_of_fold_train_scores",
        "score_type": score_type,
        **{f"oof_{key}": value for key, value in oof_metrics.items()},
        **test_metrics,
        # All 10-fold metrics are retained inside the same algorithm CSV.
        "cv_fold_results_json": json.dumps(fold_rows, sort_keys=True),
        "fit_seconds": time.perf_counter() - started,
    }


## 5. Cumulative result saving and experiment runner
Each successful method is saved atomically to the single algorithm CSV. A different data/setup fingerprint is refused rather than overwriting previous results.


In [5]:
# Fingerprint prevents mixing another dataset/setup into a previous result file.
RUN_CONTEXT = {
    "implementation": "english-all-token-v1-df463e9e",
    "data_sha256": DATA_SHA256,
    "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
    "seed": SEED, "test_size": TEST_SIZE, "cv_folds": CV_FOLDS,
    "cv_shuffle": True, "classifier": CLASSIFIER, "parameters": PARAMETERS,
    "lt_k_values": LT_K_VALUES, "gt_k_values": GT_K_VALUES,
    "filter": "strict lower < abs(sentence-local population L2 z-score) < upper",
    "token_population": "all attended tokens including special tokens; padding excluded",
    "fallback": "all attended tokens if filtered mask is empty",
    "cleaning": "notebook_34_english_clean_tweet_and_dedup",
    "train_rows": len(y_train), "test_rows": len(y_test),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "transformers": transformers.__version__},
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(RUN_CONTEXT, sort_keys=True).encode("utf-8")
).hexdigest()


def load_saved_results():
    if not RESULTS_PATH.exists():
        return pd.DataFrame()
    saved = pd.read_csv(RESULTS_PATH)
    required = {"method", "classifier", "run_signature", "macro_f1", "cv_macro_f1_mean"}
    if not required.issubset(saved.columns) or saved.empty:
        raise ValueError(f"Not a valid completed English results file: {RESULTS_PATH}")
    if not saved["run_signature"].eq(RUN_SIGNATURE).all():
        raise ValueError(
            f"{RESULTS_PATH} belongs to a different setup/dataset. "
            "It will not be overwritten. Use a new output filename for a different run."
        )
    if not saved["classifier"].eq(CLASSIFIER).all() or saved["method"].duplicated().any():
        raise ValueError("Result file has mixed algorithms or duplicate methods.")
    if not set(saved["method"]).issubset(RULE_INDEX):
        raise ValueError("Result file contains unexpected representation names.")
    return saved


def save_result(row):
    saved = load_saved_results()
    # Resume is intentional: a completed method is not retrained or duplicated.
    if not saved.empty and row["method"] in set(saved["method"]):
        return saved
    combined = pd.DataFrame(saved.to_dict("records") + [row])
    order = {item["method"]: index for index, item in enumerate(ALL_RULES)}
    combined = combined.sort_values("method", key=lambda s: s.map(order)).reset_index(drop=True)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        # Write in the destination directory so replacement is atomic.
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".tmp", prefix=RESULTS_PATH.stem + "_",
            dir=RESULTS_PATH.parent, encoding="utf-8-sig", newline="", delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            combined.to_csv(handle, index=False)
        os.replace(temporary_path, RESULTS_PATH)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()
    print(f"Saved {row['method']} -> {RESULTS_PATH.name} ({len(combined)}/61)")
    return combined


def show_results(frame):
    columns = [
        "method", "lower_k", "upper_k", "cv_macro_f1_mean", "cv_macro_f1_std",
        "accuracy", "macro_precision", "macro_recall", "macro_f1", "f1_class_1",
        "tn", "fp", "fn", "tp", "test_included_tokens", "test_included_special_tokens",
        "test_filter_fallback_texts",
    ]
    display(frame[[column for column in columns if column in frame]].reset_index(drop=True))


def run_experiment(experiment_name, items):
    requested = BASELINE_RULES + items
    saved = load_saved_results()
    completed = set(saved["method"]) if not saved.empty else set()
    pending = [item for item in requested if item["method"] not in completed]
    if pending:
        train_vectors, train_stats = extract_representations(
            train_df["clean_text"], pending, "train",
        )
        test_vectors, test_stats = extract_representations(
            test_df["clean_text"], pending, "test",
        )
        for item in pending:
            name = item["method"]
            row = evaluate_representation(item, train_vectors[name], test_vectors[name])
            row.update({
                "first_saved_in_experiment": experiment_name,
                "run_signature": RUN_SIGNATURE,
                "run_context_json": json.dumps(RUN_CONTEXT, sort_keys=True),
                "data_source": DATA_SOURCE, "data_sha256": DATA_SHA256,
                "model_name": MODEL_NAME, "max_length": MAX_LENGTH,
                "batch_size": BATCH_SIZE, "seed": SEED, "cv_folds": CV_FOLDS,
                "train_rows": len(y_train), "test_rows": len(y_test),
                "feature_dimensions": train_vectors[name].shape[1],
                "filter_operators": "strict < and >",
                "special_tokens_included": True,
                "padding_included": False,
                **{f"train_{key}": value for key, value in train_stats[name].items()},
                **{f"test_{key}": value for key, value in test_stats[name].items()},
            })
            save_result(row)
            del train_vectors[name], test_vectors[name]
        del train_vectors, test_vectors
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = load_saved_results()
    names = {item["method"] for item in requested}
    result = saved[saved["method"].isin(names)]
    print(f"{experiment_name}: {len(result)}/{len(requested)} complete")
    show_results(result)
    return result


print("Ready. Run each of the three experiment cells below.")
print("Baselines are saved once and reused in all three comparisons.")


Ready. Run each of the three experiment cells below.
Baselines are saved once and reused in all three comparisons.


## Experiment 1 — Less-than only
CLS + MEAN + 11 `mean_lt_k` settings (13 comparisons).


In [6]:
less_than_results = run_experiment("less_than", LT_RULES)


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 113.92it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting train: 3636 texts, 13 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 13 representations
  test: 640/910
  test: 910/910
  cls fold 1/10: macro_f1=0.785409
  cls fold 2/10: macro_f1=0.770068
  cls fold 3/10: macro_f1=0.805536
  cls fold 4/10: macro_f1=0.796278
  cls fold 5/10: macro_f1=0.818572
  cls fold 6/10: macro_f1=0.825025
  cls fold 7/10: macro_f1=0.791336
  cls fold 8/10: macro_f1=0.786668
  cls fold 9/10: macro_f1=0.774532
  cls fold 10/10: macro_f1=0.837366
Saved cls -> english_all_tokens_rf_results.csv (1/61)
  mean fold 1/10: macro_f1=0.818815
  mean fold 2/10: macro_f1=0.857255
  mean fold 3/10: macro_f1=0.845421
  mean fold 4/10: macro_f1=0.859730
  mean fold 5/10: macro_f1=0.823672
  mean fold 6/10: macro_f1=0.842308
  mean fold 7/10: macro_f1=0.857153
  mean fold 8/10: macro_f1=0.814088
  mean fold 9/10: macro_f1=0.830263
  mean fold 10/10: macro_f1=0.

,method,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,NaN,NaN,0.799079,0.022145,0.832967,0.758419,0.794190,0.772902,0.656109,613,97,55,145,910,910,0
1,mean,NaN,NaN,0.840718,0.017745,0.883516,0.831383,0.826585,0.828943,0.732323,659,51,55,145,15911,1820,0
2,mean_lt_0.5,NaN,0.5,0.815617,0.019108,0.858242,0.793818,0.790634,0.792205,0.675063,647,63,66,134,5709,866,2
3,mean_lt_1,NaN,1.0,0.845827,0.015163,0.881319,0.825429,0.832359,0.828806,0.733990,653,57,51,149,10328,1417,0
4,mean_lt_1.5,NaN,1.5,0.845086,0.015277,0.892308,0.848423,0.830423,0.838887,0.746114,668,42,56,144,13706,1725,0
5,mean_lt_2,NaN,2.0,0.840675,0.017296,0.884615,0.833333,0.827289,0.830247,0.734177,660,50,55,145,15401,1807,0
6,mean_lt_2.5,NaN,2.5,0.843695,0.017279,0.870330,0.807261,0.832500,0.818585,0.721698,639,71,47,153,15829,1817,0
7,mean_lt_3,NaN,3.0,0.839641,0.016941,0.872527,0.812500,0.821338,0.816767,0.715686,648,62,54,146,15901,1820,0
8,mean_lt_3.5,NaN,3.5,0.841360,0.017542,0.884615,0.835439,0.821901,0.828351,0.730077,663,47,58,142,15910,1820,0
9,mean_lt_4,NaN,4.0,0.840718,0.017745,0.883516,0.831383,0.826585,0.828943,0.732323,659,51,55,145,15911,1820,0


## Experiment 2 — Greater-than only
CLS + MEAN + 4 `mean_gt_k` settings (6 comparisons). Baselines are reused from the same file.


In [7]:
greater_than_results = run_experiment("greater_than", GT_RULES)


Extracting train: 3636 texts, 4 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 4 representations
  test: 640/910
  test: 910/910
  mean_gt_0.0001 fold 1/10: macro_f1=0.823769
  mean_gt_0.0001 fold 2/10: macro_f1=0.862132
  mean_gt_0.0001 fold 3/10: macro_f1=0.847344
  mean_gt_0.0001 fold 4/10: macro_f1=0.855362
  mean_gt_0.0001 fold 5/10: macro_f1=0.823769
  mean_gt_0.0001 fold 6/10: macro_f1=0.837065
  mean_gt_0.0001 fold 7/10: macro_f1=0.865556
  mean_gt_0.0001 fold 8/10: macro_f1=0.815614
  mean_gt_0.0001 fold 9/10: macro_f1=0.828816
  mean_gt_0.0001 fold 10/10: macro_f1=0.859610
Saved mean_gt_0.0001 -> english_all_tokens_rf_results.csv (14/61)
  mean_gt_0.001 fold 1/10: macro_f1=0.803235
  mean_gt_0.001 fold 2/10: macro_f1=0.854497
  mean_gt_0.001 fold 3/10: macro_f1=0.847799
  mean_gt_0.001 fold 4/10: macro_f1=0.854185
  mean_gt_0.001 fold 5/10: macro_f1=0.827010
  mean_gt

,method,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,NaN,NaN,0.799079,0.022145,0.832967,0.758419,0.794190,0.772902,0.656109,613,97,55,145,910,910,0
1,mean,NaN,NaN,0.840718,0.017745,0.883516,0.831383,0.826585,0.828943,0.732323,659,51,55,145,15911,1820,0
2,mean_gt_0.0001,0.0001,NaN,0.841904,0.018368,0.883516,0.832702,0.822993,0.827681,0.729592,661,49,57,143,15909,1820,0
3,mean_gt_0.001,0.0010,NaN,0.840811,0.018432,0.868132,0.804300,0.829296,0.815511,0.716981,638,72,48,152,15903,1820,0
4,mean_gt_0.01,0.0100,NaN,0.837519,0.017940,0.872527,0.810926,0.830317,0.819875,0.722488,643,67,49,151,15794,1800,0
5,mean_gt_0.1,0.1000,NaN,0.838587,0.013116,0.875824,0.817800,0.823451,0.820564,0.720988,651,59,54,146,14724,1631,0


## Experiment 3 — Greater-than AND less-than
CLS + MEAN + all 44 `mean_gt_lower_lt_upper` settings (46 comparisons). Baselines are reused.


In [8]:
combined_results = run_experiment("greater_and_less_than", BAND_RULES)


Extracting train: 3636 texts, 44 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 44 representations
  test: 640/910
  test: 910/910
  mean_gt_0.0001_lt_0.5 fold 1/10: macro_f1=0.777370
  mean_gt_0.0001_lt_0.5 fold 2/10: macro_f1=0.801695
  mean_gt_0.0001_lt_0.5 fold 3/10: macro_f1=0.839005
  mean_gt_0.0001_lt_0.5 fold 4/10: macro_f1=0.834320
  mean_gt_0.0001_lt_0.5 fold 5/10: macro_f1=0.814237
  mean_gt_0.0001_lt_0.5 fold 6/10: macro_f1=0.802947
  mean_gt_0.0001_lt_0.5 fold 7/10: macro_f1=0.838518
  mean_gt_0.0001_lt_0.5 fold 8/10: macro_f1=0.804755
  mean_gt_0.0001_lt_0.5 fold 9/10: macro_f1=0.807937
  mean_gt_0.0001_lt_0.5 fold 10/10: macro_f1=0.831373
Saved mean_gt_0.0001_lt_0.5 -> english_all_tokens_rf_results.csv (18/61)
  mean_gt_0.0001_lt_1 fold 1/10: macro_f1=0.833525
  mean_gt_0.0001_lt_1 fold 2/10: macro_f1=0.869464
  mean_gt_0.0001_lt_1 fold 3/10: macro_f1=0.848359
  

,method,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,NaN,NaN,0.799079,0.022145,0.832967,0.758419,0.794190,0.772902,0.656109,613,97,55,145,910,910,0
1,mean,NaN,NaN,0.840718,0.017745,0.883516,0.831383,0.826585,0.828943,0.732323,659,51,55,145,15911,1820,0
2,mean_gt_0.0001_lt_0.5,0.0001,0.5,0.815216,0.020177,0.836264,0.761749,0.789120,0.773510,0.654292,620,90,59,141,5707,866,2
3,mean_gt_0.0001_lt_1,0.0001,1.0,0.844770,0.016272,0.880220,0.826807,0.820880,0.823780,0.724051,658,52,57,143,10326,1417,0
4,mean_gt_0.0001_lt_1.5,0.0001,1.5,0.847555,0.015634,0.889011,0.840581,0.831901,0.836112,0.743003,663,47,54,146,13704,1725,0
5,mean_gt_0.0001_lt_2,0.0001,2.0,0.840419,0.018731,0.886813,0.835930,0.832289,0.834086,0.740554,660,50,53,147,15399,1807,0
6,mean_gt_0.0001_lt_2.5,0.0001,2.5,0.843092,0.018925,0.867033,0.802597,0.830387,0.814882,0.716628,636,74,47,153,15827,1817,0
7,mean_gt_0.0001_lt_3,0.0001,3.0,0.842172,0.016092,0.871429,0.810430,0.822430,0.816145,0.715328,646,64,53,147,15899,1820,0
8,mean_gt_0.0001_lt_3.5,0.0001,3.5,0.841446,0.018549,0.884615,0.834707,0.823697,0.828990,0.731458,662,48,57,143,15908,1820,0
9,mean_gt_0.0001_lt_4,0.0001,4.0,0.841904,0.018368,0.883516,0.832702,0.822993,0.827681,0.729592,661,49,57,143,15909,1820,0


## All options — one algorithm file
Displays all saved methods in the requested baseline/LT/GT/band order. Check that all 61 are present before interpreting the complete experiment.


In [9]:
all_results = load_saved_results()
completed_methods = set(all_results["method"]) if not all_results.empty else set()
missing_methods = [item["method"] for item in ALL_RULES if item["method"] not in completed_methods]
print("Saved results:", RESULTS_PATH)
print(f"Completed: {len(completed_methods)}/61")
if missing_methods:
    print("Incomplete — still missing:", missing_methods)
else:
    print("All three experiments are complete.")
show_results(all_results)


Saved results: <PROJECT_ROOT>\results\english_all_tokens_rf_results.csv
Completed: 61/61
All three experiments are complete.


,method,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,NaN,NaN,0.799079,0.022145,0.832967,0.758419,0.794190,0.772902,0.656109,613,97,55,145,910,910,0
1,mean,NaN,NaN,0.840718,0.017745,0.883516,0.831383,0.826585,0.828943,0.732323,659,51,55,145,15911,1820,0
2,mean_lt_0.5,NaN,0.5,0.815617,0.019108,0.858242,0.793818,0.790634,0.792205,0.675063,647,63,66,134,5709,866,2
3,mean_lt_1,NaN,1.0,0.845827,0.015163,0.881319,0.825429,0.832359,0.828806,0.733990,653,57,51,149,10328,1417,0
4,mean_lt_1.5,NaN,1.5,0.845086,0.015277,0.892308,0.848423,0.830423,0.838887,0.746114,668,42,56,144,13706,1725,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,mean_gt_0.1_lt_3.5,0.1,3.5,0.836658,0.013628,0.871429,0.809821,0.826021,0.817399,0.718072,644,66,51,149,14723,1631,0
57,mean_gt_0.1_lt_4,0.1,4.0,0.838587,0.013116,0.875824,0.817800,0.823451,0.820564,0.720988,651,59,54,146,14724,1631,0
58,mean_gt_0.1_lt_4.5,0.1,4.5,0.838587,0.013116,0.875824,0.817800,0.823451,0.820564,0.720988,651,59,54,146,14724,1631,0
59,mean_gt_0.1_lt_5,0.1,5.0,0.838587,0.013116,0.875824,0.817800,0.823451,0.820564,0.720988,651,59,54,146,14724,1631,0
